# Week 1 Synthesis: AI Agent Systems, RAG, Grounding, and Safety

## 1. Different AI Agent Loops

### 1. ChatGPT-style Conversational Assistant

```text
User
    ↓
Reason / Plan
    ↓
(Optional) Tool Calls
    ↓
Generate Response
    ↓
Safety Check
    ↓
User
```

**Use case**
- General-purpose assistant
- Multi-turn conversations
- Tool-augmented reasoning

---

### 2. Perplexity-style Search Assistant

```text
User Query
    ↓
Retrieve Documents
    ↓
Re-rank
    ↓
Context Assembly
    ↓
LLM
    ↓
Answer + Public Citations
```

**Key idea**
- Retrieval-first
- Every factual claim should ideally be grounded in retrieved sources.

---

### 3. Cursor Agent

```text
User Request
    ↓
Plan
    ↓
Generate Diff
    ↓
Apply Changes
    ↓
Type Check
    ↓
Lint
    ↓
Tests
    ↓
Build
    ↓
Smoke Test
    ↓
User Review
```

**Key idea**
- Multi-file editing
- Continuous verification before completion

---

### 4. GitHub Copilot

```text
Open Files
+
Repository Index
+
Embeddings
        ↓
Context Assembly
        ↓
Code Completion
        ↓
Developer Accepts / Rejects
```

**Key idea**
- Code-context retrieval
- Function/class-aware chunking
- Repository-aware suggestions

---

### 5. Lovable / Bolt

```text
Prompt
    ↓
Generate Full Application
    ↓
Run
    ↓
Fix Errors
    ↓
Repeat Until Working
```

**Key idea**
- Autonomous

- Full-stack application generation
- Verification loop continues until the application is functional

---

### 6. Structured Enterprise AI Pipeline

```text
User Query
      ↓
Intent Classification
      ↓
Access Control
      ↓
Retrieve Documents
      ↓
Call Internal Tools
      ↓
Context Assembly
      ↓
LLM
      ↓
Output Validation
      ↓
User
```

**Key idea**
- The LLM is only one component.
- Most reliability comes from the surrounding system.

---

# 2. ReAct vs Structured Pipeline

## ReAct Loop

```text
Think
  ↓
Act
  ↓
Observe
  ↓
Think Again
  ↓
Answer
```

### Characteristics

- Dynamic
- Agent decides which tools to call
- Flexible
- Good for open-ended reasoning
- Less predictable

Example:

```text
User:
"My invoice looks wrong."

↓

Search billing docs

↓

Call invoice API

↓

Observe result

↓

Reason

↓

Respond
```

---

## Structured Pipeline

```text
Intent
    ↓
Retrieve
    ↓
Tool Calls
    ↓
Context Assembly
    ↓
LLM
    ↓
Validation
```

### Characteristics

- Predictable
- Easier to test
- Easier to audit
- Better for production SaaS
- Preferred where correctness matters

---

# 3. Why We Use RAG

Without RAG:

```text
Entire documentation
        ↓
LLM
```

Problems:

- High token cost
- Poor context quality
- Context window limitations
- More hallucinations

---

With RAG:

```text
User Query
      ↓
Embedding
      ↓
Vector Search
      ↓
Relevant Chunks
      ↓
LLM
```

Benefits:

- Lower token usage
- Better grounding
- Better factual accuracy
- Faster inference
- Easier document updates

---

# 4. Context Assembly

The LLM should receive only the context required for the current query.

Possible context includes:

- Retrieved documentation
- Product state
- User plan
- Account metadata
- Tool outputs
- Conversation history

Example:

```text
Billing Question

↓

Retrieve:

✓ Billing FAQ
✓ Current plan
✓ Invoice status

Do NOT send:

✗ Entire company documentation
✗ Previous unrelated conversations
✗ All user metadata
```

---

# 5. Tool Calls vs Documentation

These are different.

## Documentation

Usually contains:

- Help articles
- Policies
- FAQs
- Product documentation

Mostly static.

---

## Tool Calls

Provide live information.

Examples:

- Current subscription
- Order status
- Invoice
- User permissions
- Calendar events
- CRM information

Tools are the source of live truth.

---

# 6. Public Documentation vs Internal Knowledge

This distinction is critical.

## Public Documentation

Can contain:

- FAQs
- User guides
- Product documentation

Safe to expose.

---

## Internal Knowledge

Can contain:

- Support playbooks
- Internal policies
- Escalation procedures
- Internal prompts
- Debug information
- Employee notes

Should never be exposed directly.

---

# 7. Why We Should Not Trust the LLM

The LLM is **not** the source of truth.

It is:

- a language generator
- a reasoning engine

It should not decide:

- permissions
- visibility
- truth
- user authorization

Those decisions belong to the backend.

---

# 8. Structured Outputs

Internal systems should preferably return structured outputs instead of raw text.

Example:

```json
{
  "plan": "Pro",
  "seats_used": 5,
  "seat_limit": 5,
  "allowed_user_message": "You have reached the limit of your current plan."
}
```

Advantages:

- Easier validation
- Easier testing
- Lower hallucination risk
- Easier backend integration

---

# 9. Can Structured Output Alone Prevent Hallucinations?

No.

This is an important realization.

Example:

LLM outputs:

```json
{
  "title": "Billing FAQ",
  "visibility": "public"
}
```

The JSON format is valid.

But the content may still be wrong.

Structured output guarantees **format**, not **correctness**.

---

# 10. Where Truth Should Come From

The backend owns the truth.

Example:

```json
{
    "doc_id": "billing_123",
    "visibility": "public",
    "title": "Billing FAQ",
    "content": "...",
    "url": "..."
}
```

The backend/database decides:

- document title
- visibility
- permissions
- ownership

The LLM should not invent these fields.

---

# 11. Safety Architecture

Never rely on prompt instructions like:

```text
"Do not reveal internal information."
```

Instead, enforce safety in multiple layers.

```text
User
      ↓
Intent Classification
      ↓
Authentication
      ↓
Authorization
      ↓
Access-Controlled Retrieval
      ↓
Tool Calls
      ↓
Context Assembly
      ↓
LLM
      ↓
Output Validation
      ↓
User
```

Security should be enforced **outside** the model.

---

# 12. Rule-Based Safety Checks

Examples:

## Access Control

Only retrieve documents the user is allowed to access.

---

## Tenant Isolation

Prevent users from accessing another company's data.

---

## Field Allowlist

Only expose fields explicitly marked as safe.

Example:

```text
✓ plan_name
✓ invoice_status
✓ order_status

✗ fraud_score
✗ internal_notes
✗ support_comments
```

---

## Secret Detection

Block responses containing:

- API keys
- Tokens
- Internal prompts
- Credentials
- Hidden instructions

---

## Claim Verification

Compare generated claims with tool outputs.

Example:

Tool:

```text
Plan = Pro
```

LLM:

```text
Enterprise
```

Mismatch → Block or regenerate.

---

# 13. Public Citations

Citations are useful **only** when the cited document is public.

Examples:

- Billing FAQ
- User documentation
- Setup guides
- API documentation

Do **not** cite:

- Internal playbooks
- Private policies
- System prompts
- Employee documentation

---

# 14. Should Every SaaS Assistant Show Citations?

No.

Examples where citations add little value:

- Current subscription
- Order tracking
- Payment status
- Account balance

These answers come from verified APIs, not documents.

Examples where citations help:

- Refund policy
- Password reset guide
- API documentation
- Product setup instructions

---

# 15. Cost vs Security

Security introduces additional work, but it should be designed efficiently.

Instead of validating everything with another LLM:

```text
Use cheap deterministic checks first.

↓

Escalate to expensive validation only when necessary.
```

Examples:

Low-risk:

```text
Public FAQ
↓

Small model

↓

Light validation
```

High-risk:

```text
Internal tools

↓

Structured output

↓

Stronger reasoning

↓

Strict validation
```

---

# 16. What Happens If Every LLM Hallucinates?

This is the worst-case scenario.

The production system should **not** continue guessing.

Fallback should be deterministic.

Example:

```text
"I'm not confident enough to answer this safely.

I'll connect you with our support team."
```

or

```text
"This request requires manual review."
```

Never allow the system to fabricate answers when confidence is low.

---

# 17. Final Mental Models

## RAG

```text
Retrieve first.

Generate second.
```

---

## Production AI

```text
LLM is not the system.

LLM is one component inside the system.
```

---

## Truth

```text
Backend owns truth.

Database stores truth.

Tools return truth.

LLM explains truth.
```

---

## Security

```text
Never depend on model obedience.

Enforce security before and after the model.
```

---

## Structured Outputs

```text
Structured outputs guarantee format.

They do NOT guarantee correctness.
```

---

## Enterprise AI Principle

A production AI assistant should be designed so that:

- The LLM is **not trusted** as the source of truth.
- Permissions are enforced by the backend.
- Retrieval respects access control.
- Internal data is filtered before reaching the model.
- Tool outputs are structured and verifiable.
- Responses are validated before being shown to the user.
- When confidence is low or validation fails, the system falls back to deterministic behavior or escalates to a human instead of guessing.